In [2]:
#2021.10.20. TUE
#Team_SmokeFree

## Data Preprocessing For MCLP 
#00. 패키지 호출하기. 
import warnings
import re
import pandas as pd 
import numpy as np 
import geopandas as gpd
from fiona.crs import from_epsg
import requests

#00-1. warning message ignore 
warnings.filterwarnings(action='ignore')

#01. kakao_api_key.txt 파일 부르기. 
with open("A:/Python_Project/Team_SmokeFree/key/kakao_api_key.txt",mode="r") as key_file :
    kakao_api_key = key_file.read()


In [3]:
#01. RANDOM_POINT 데이터셋 전처리하기. 
#(1) RANDOM_POINT 데이터셋 불러오기. 
RANDOM_POINT = pd.read_csv('temp_file_03.csv', encoding='utf-8')

#(2) 처리 결과 확인하기. 
RANDOM_POINT

,rand_point,SGG_NM,geometry
0,0,금천구,POINT (126.9134055885386 37.44996180218595)
1,1,금천구,POINT (126.9143943524618 37.44996374526017)
2,2,금천구,POINT (126.9142583604479 37.44993860398068)
3,3,금천구,POINT (126.9132887655418 37.44995610565504)
4,4,금천구,POINT (126.9136375368365 37.45000822498514)
...,...,...,...
40286,41191,동작구,POINT (126.9536105492901 37.49994690939378)
40287,41192,동작구,POINT (126.9536832343939 37.50027154889482)
40288,41193,동작구,POINT (126.9537327787121 37.50010520158163)
40289,41194,동작구,POINT (126.9537081199655 37.50054022273152)


In [4]:
#(3) 위도, 경도 추출하기. 
RANDOM_POINT['위도'] = np.nan
RANDOM_POINT['경도'] = np.nan

for i in range(len(RANDOM_POINT)) :
    RANDOM_POINT['경도'][i] = float(re.sub('[^0-9.]','', RANDOM_POINT['geometry'][i].split(' ')[1]))
    RANDOM_POINT['위도'][i] = float(re.sub('[^0-9.]','', RANDOM_POINT['geometry'][i].split(' ')[2]))

#(4) 처리 결과 확인하기. 
RANDOM_POINT

,rand_point,SGG_NM,geometry,위도,경도
0,0,금천구,POINT (126.9134055885386 37.44996180218595),37.449962,126.913406
1,1,금천구,POINT (126.9143943524618 37.44996374526017),37.449964,126.914394
2,2,금천구,POINT (126.9142583604479 37.44993860398068),37.449939,126.914258
3,3,금천구,POINT (126.9132887655418 37.44995610565504),37.449956,126.913289
4,4,금천구,POINT (126.9136375368365 37.45000822498514),37.450008,126.913638
...,...,...,...,...,...
40286,41191,동작구,POINT (126.9536105492901 37.49994690939378),37.499947,126.953611
40287,41192,동작구,POINT (126.9536832343939 37.50027154889482),37.500272,126.953683
40288,41193,동작구,POINT (126.9537327787121 37.50010520158163),37.500105,126.953733
40289,41194,동작구,POINT (126.9537081199655 37.50054022273152),37.500540,126.953708


In [5]:
#(5) 다음 작업을 위한 복사본 저장하기. 
RANDOM_POINT_no_HGD = RANDOM_POINT

In [6]:
# API LIMIT TEST 
# local_url = "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
# x = 126.973899
# y = 37.5701764
# url=f'{local_url}?x={x}&y={y}&input_coord=WGS84' 
# REQ_RESULT = requests.get(url, headers={"Authorization":f"KakaoAK {kakao_api_key}"}).json()
# REQ_RESULT

{'meta': {'total_count': 2},
 'documents': [{'region_type': 'B',
   'code': '1111012000',
   'address_name': '서울특별시 종로구 신문로1가',
   'region_1depth_name': '서울특별시',
   'region_2depth_name': '종로구',
   'region_3depth_name': '신문로1가',
   'region_4depth_name': '',
   'x': 126.97390161420634,
   'y': 37.57017314981071},
  {'region_type': 'H',
   'code': '1111053000',
   'address_name': '서울특별시 종로구 사직동',
   'region_1depth_name': '서울특별시',
   'region_2depth_name': '종로구',
   'region_3depth_name': '사직동',
   'region_4depth_name': '',
   'x': 126.96884605608865,
   'y': 37.57618696588561}]}

In [9]:
#03. API 이용하여 정보 추출하기. 
#(1) 주소지를 기반으로 행정동 코드, 경도, 위도 추출하여 리스트에 추가하기. 
RANDOM_POINT_no_HGD_x       = RANDOM_POINT_no_HGD['경도'].values
RANDOM_POINT_no_HGD_y       = RANDOM_POINT_no_HGD['위도'].values
RANDOM_POINT_no_HGD_자치구      = []
RANDOM_POINT_no_HGD_행정동_코드 = []

local_url = "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
for num in range(len(RANDOM_POINT_no_HGD_x))                                                        :
    try                                                                                             :
        x = RANDOM_POINT_no_HGD_x[num]
        y = RANDOM_POINT_no_HGD_y[num]   
        url=f'{local_url}?x={x}&y={y}&input_coord=WGS84' 
    except Exception                                                                                :
        x = RANDOM_POINT_no_HGD_x[num]
        y = RANDOM_POINT_no_HGD_y[num]      
        url=f'{local_url}?x={x}&y={y}&input_coord=WGS84' 
    try                                                                                             : 
        REQ_RESULT = requests.get(url, headers={"Authorization":f"KakaoAK {kakao_api_key}"}).json()
        RANDOM_POINT_no_HGD_자치구.append(REQ_RESULT['documents'][1]['region_2depth_name'])
        RANDOM_POINT_no_HGD_행정동_코드.append(REQ_RESULT['documents'][1]['code'])
        print(f'DONE ! index_num = {num}')
    except Exception                                                                                :
        try                                                                                         :
            x = RANDOM_POINT_no_HGD_x[num]
            y = RANDOM_POINT_no_HGD_y[num]   
            url=f'{local_url}?x={x}&y={y}&input_coord=WGS84' 
            REQ_RESULT = requests.get(url, headers={"Authorization":f"KakaoAK {kakao_api_key}"}).json()
            RANDOM_POINT_no_HGD_자치구.append(REQ_RESULT['documents'][1]['region_2depth_name'])
            RANDOM_POINT_no_HGD_행정동_코드.append(REQ_RESULT['documents'][1]['code'])
            print(f'DONE ! index_num = {num}')
        except Exception                                                                            :
            RANDOM_POINT_no_HGD_자치구.append('ERROR')
            RANDOM_POINT_no_HGD_행정동_코드.append('ERROR')
            print(f'ERROR ! index_num = {num}')

DONE ! index_num = 0
DONE ! index_num = 1
DONE ! index_num = 2
DONE ! index_num = 3
DONE ! index_num = 4
DONE ! index_num = 5
DONE ! index_num = 6
DONE ! index_num = 7
DONE ! index_num = 8
DONE ! index_num = 9
DONE ! index_num = 10
DONE ! index_num = 11
DONE ! index_num = 12
DONE ! index_num = 13
DONE ! index_num = 14
DONE ! index_num = 15
DONE ! index_num = 16
DONE ! index_num = 17
DONE ! index_num = 18
DONE ! index_num = 19
DONE ! index_num = 20
DONE ! index_num = 21
DONE ! index_num = 22
DONE ! index_num = 23
DONE ! index_num = 24
DONE ! index_num = 25
DONE ! index_num = 26
DONE ! index_num = 27
DONE ! index_num = 28
DONE ! index_num = 29
DONE ! index_num = 30
DONE ! index_num = 31
DONE ! index_num = 32
DONE ! index_num = 33
DONE ! index_num = 34
DONE ! index_num = 35
DONE ! index_num = 36
DONE ! index_num = 37
DONE ! index_num = 38
DONE ! index_num = 39
DONE ! index_num = 40
DONE ! index_num = 41
DONE ! index_num = 42
DONE ! index_num = 43
DONE ! index_num = 44
DONE ! index_num = 4

In [11]:
#(2) 값 추가하기.
RANDOM_POINT_no_HGD['자치구']      = RANDOM_POINT_no_HGD_자치구
RANDOM_POINT_no_HGD['행정동_코드'] = RANDOM_POINT_no_HGD_행정동_코드

#(5) 처리 결과 확인하기. 
RANDOM_POINT_no_HGD

,rand_point,SGG_NM,geometry,위도,경도,자치구,행정동_코드
0,0,금천구,POINT (126.9134055885386 37.44996180218595),37.449962,126.913406,금천구,1154568000
1,1,금천구,POINT (126.9143943524618 37.44996374526017),37.449964,126.914394,금천구,1154568000
2,2,금천구,POINT (126.9142583604479 37.44993860398068),37.449939,126.914258,금천구,1154568000
3,3,금천구,POINT (126.9132887655418 37.44995610565504),37.449956,126.913289,금천구,1154568000
4,4,금천구,POINT (126.9136375368365 37.45000822498514),37.450008,126.913638,금천구,1154568000
...,...,...,...,...,...,...,...
40286,41191,동작구,POINT (126.9536105492901 37.49994690939378),37.499947,126.953611,동작구,1159053000
40287,41192,동작구,POINT (126.9536832343939 37.50027154889482),37.500272,126.953683,동작구,1159053000
40288,41193,동작구,POINT (126.9537327787121 37.50010520158163),37.500105,126.953733,동작구,1159053000
40289,41194,동작구,POINT (126.9537081199655 37.50054022273152),37.500540,126.953708,동작구,1159053000


In [13]:
#(6) 처리 결과 저장하기. 
RANDOM_POINT_no_HGD.to_excel('temp_file.xlsx', index=False)